# 梯度累积、裁剪与训练稳定性

## 学习目标

实现梯度累积和梯度裁剪，正确安排 optimizer 与 scheduler 的调用时机，并通过梯度范数发现不稳定训练。

## 概念模型

梯度累积模拟更大的有效 batch；梯度裁剪限制更新前的梯度范数。两者都不能修复错误数据、错误损失函数或不合理学习率。

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
x = torch.randn(40, 6)
y = (x[:, :3].sum(dim=1) > 0).long()
loader = DataLoader(TensorDataset(x, y), batch_size=4, shuffle=False)
model = nn.Sequential(nn.Linear(6, 16), nn.ReLU(), nn.Linear(16, 2))
optimizer = torch.optim.AdamW(model.parameters(), lr=0.02)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.5)
loss_fn = nn.CrossEntropyLoss()

### 实验 1：梯度累积

**实验目的**：每两个 micro-batch 执行一次 optimizer step，模拟更大的有效 batch。loss 除以 `accumulation_steps`，使梯度尺度接近对大 batch 平均 loss 的一次反向。

只有 step 边界才清梯度；若 batch 数不能整除累积步数，还必须处理末尾剩余梯度。BatchNorm 统计仍按 micro-batch 更新，因此不完全等价于真实大 batch。


In [ ]:
accumulation_steps = 2
optimizer.zero_grad(set_to_none=True)
updates = 0
for batch_index, (batch_x, batch_y) in enumerate(loader, 1):
    loss = loss_fn(model(batch_x), batch_y) / accumulation_steps
    loss.backward()
    if batch_index % accumulation_steps == 0 or batch_index == len(loader):
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        updates += 1
print('micro-batches:', len(loader), 'optimizer updates:', updates)
assert updates == 5

### 实验 2：梯度范数与裁剪

**实验目的**：人为放大 loss 产生大梯度，再将全局范数裁剪到 1。`clip_grad_norm_` 返回裁剪前范数，裁剪后应重新计算或检查上限。

裁剪限制更新幅度，但不修复 NaN、错误 loss 或不合理学习率。AMP 下应先 unscale 再裁剪。


In [ ]:
optimizer.zero_grad(set_to_none=True)
batch_x, batch_y = next(iter(loader))
large_loss = loss_fn(model(batch_x), batch_y) * 1000
large_loss.backward()
before_clip = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
after_clip = torch.sqrt(sum(parameter.grad.square().sum() for parameter in model.parameters()))
print('gradient norm:', float(before_clip), '->', float(after_clip))
assert after_clip <= 1.0001
optimizer.step()

### 实验 3：scheduler 调用时机和有限值检查

**实验目的**：在规定边界调用 scheduler，并检查参数和梯度均为有限值。不同 scheduler 按 batch、epoch 或验证指标调用，不能统一套用同一位置。

有限值检查应尽量靠近首次产生异常的位置；只在 epoch 末检查会丢失定位信息。


In [ ]:
old_lr = optimizer.param_groups[0]['lr']
scheduler.step()
new_lr = optimizer.param_groups[0]['lr']
assert new_lr < old_lr
for parameter in model.parameters():
    assert torch.isfinite(parameter).all()
    if parameter.grad is not None:
        assert torch.isfinite(parameter.grad).all()
print('learning rate:', old_lr, '->', new_lr)

## 检查点

写出梯度累积时清梯度、反向、裁剪和更新的顺序；解释为什么累积时通常需要缩放 loss；说明不同 scheduler 为什么调用时机不同。

## 试一试

比较真实 batch size 8 与 batch size 4、累积 2 步的参数更新；故意去掉 loss 缩放并观察梯度范数。

## 常见错误与调试

每个 micro-batch 都清梯度、忘记处理最后不足累积步数的 batch、在 backward 前裁剪、AMP 下未先 unscale 就裁剪、scheduler 调用频率错误。